## Building the Allegheny County polluter point dataset

1. **Pick the analysis year**
   - Source: EasyRSEI dashboard (edap.epa.gov)
   - RSEI usually lags TRI by a year or more. Check the most recent reporting year EasyRSEI covers, then use that same year for TRI.
   - Record the year for the methods section.

2. **Download the TRI file**
   - Source: EPA TRI Basic Data Files page (epa.gov/toxics-release-inventory-tri-program)
   - Select your year and Pennsylvania, then download the CSV.
   - Also download the "Guide to Using the TRI Basic Data Files" PDF. Column names include number prefixes that have changed across years.
   - Each row is one facility-chemical combination, not one facility.

3. **Filter and collapse TRI to facilities**
   - Filter county to Allegheny (check the exact capitalization in the file).
   - Keep: TRI facility ID, facility name, city, latitude, longitude, industry/NAICS, parent company, fugitive air, stack air.
   - Drop dioxin rows from the pounds totals (dioxin is reported in grams).
   - Create `total_air` = fugitive air + stack air.
   - Group by TRI facility ID: take the first name, city, and coordinates; sum `total_air`.
   - Drop facilities with zero air releases.
   - Confirm one row per facility ID.

4. **Export RSEI Scores**
   - Source: EasyRSEI dashboard
   - Set the value option to RSEI Score; filter to your year, Pennsylvania, and Allegheny County (use city or ZIP filters if county isn't available).
   - Export the facility-level table with facility ID, name, and Score.
   - Optional: repeat with RSEI Hazard.
   - Before exporting, confirm the table includes the TRI facility ID. If it only has names, match by name in step 5.

5. **Join TRI and RSEI**
   - Left join the RSEI export onto the TRI facility table on TRI facility ID.
   - Check unmatched rows in both directions.
   - TRI facility with no Score: usually released only chemicals without RSEI toxicity weights. Set Score to 0 and note it in the report.
   - RSEI facility with no TRI match: usually an ID-format mismatch (whitespace, numeric vs. string).

6. **Rank and select facilities**
   - Sort by RSEI Score, descending.
   - Keep every facility; add a `rank` column, a `top10` flag, and a `us_steel` flag (from parent company or facility name).
   - Sanity check: Clairton and Edgar Thomson should appear near the top. If not, recheck the year, filter, or join.

7. **Build and check geometry**
   - Create point geometry from longitude (x) and latitude (y), in that order.
   - Set CRS to EPSG:4326, then reproject to EPSG:2272.
   - Plot over the Allegheny County boundary: every point should fall inside the county, and Clairton should land on the river in Clairton.
   - Fix or drop facilities with missing or wrong coordinates, and document it.

8. **Save and hand off**
   - Save as GeoPackage or GeoJSON in EPSG:2272.
   - Columns: facility ID, name, city, parent company, total air (lbs), RSEI Score, RSEI Hazard, rank, top-10 flag, U.S. Steel flag.
   - Record the year, selection rule, and join results for the report's data section.